# Introduction to NumPy

## What is NumPy?
NumPy is a fundamental package for scientific computing in Python. It provides support for arrays, matrices, and many mathematical functions that are useful in data analysis. In the field of geomatics, NumPy is particularly valuable for handling large datasets, performing numerical calculations, and integrating with other Python libraries for geospatial data analysis.

## Importance of NumPy in Geomatics and Data Analysis
In geomatics, you often work with large datasets, such as raster images, point clouds, or tabular data with coordinates. NumPy makes it easy to perform efficient computations on these datasets, whether you're performing basic arithmetic operations, calculating distances between points, or transforming coordinates.

This tutorial will guide you through the basics of NumPy and demonstrate how it can be applied to common tasks in geomatics.

NumPy runs behind the scenes in pandas. Pandas is built on top of NumPy and heavily relies on NumPy's data structures and functions to operate efficiently. NumPy is generally better suited than pandas for working with rasters due to efficiency.

## Creating NumPy Arrays
Let's start by creating a simple NumPy array

In [ ]:
# NumPy is nearly always imported with the short nickname 'np'
import numpy as np

In [ ]:
# np.array() turns an ordinary Python list into a NumPy array
array_1d = np.array([1, 2, 3, 4, 5])

In [ ]:
# Notice that a NumPy array prints without commas between the values
print("1D Array:", array_1d)

There are several ways you can select elements in an array. When using square brackets, the selection works just like it does for a Python list.

In [ ]:
# Index 0 gives the first element
array_1d[0]

In [ ]:
# Index 1 gives the second element
array_1d[1]

In [ ]:
# Index -1 gives the last element
array_1d[-1]

Let's create a 2D array!

In [ ]:
# Creating a 2D array (e.g., representing coordinates in a 2D space)
array_2d = np.array([[1, 2], [3, 4], [5, 6]])
print("2D Array:\n", array_2d)

In [ ]:
# Every NumPy array has a data type (dtype) telling us what kind of
# numbers it holds. All values in one array share the same type.
print("Data type of array_1d:", array_1d.dtype)


In [ ]:
# For a 2D array, a single index gives back a whole row
array_2d[0]

Let's simulate a raster image now.

In [ ]:
# Define the dimensions of the raster (e.g., 100x100 pixels)
rows, cols = 100, 100

# Create an empty raster (initialized with zeros, representing a black image)
raster = np.zeros((rows, cols), dtype=np.uint8)
raster

Let's say we want to change the values in the first few rows of the raster. We can do that with slicing.

A slice like `raster[0:3, 0:70]` selects the first three **rows** and the first 70 **columns** of the raster. The pattern is `array[row_slice, column_slice]`, with the row slice first.



In [ ]:
# Set a block of pixels to 255 (white): the first 3 rows, first 70 columns
raster[0:3, 0:70] = 255  

print(raster)

Using indexing and slicing we can access a single value or a set of values in the array.

In [ ]:

# A single pixel: row 0, column 0
print("First element of raster:", raster[0,0])


In [ ]:
# A whole row. The bare colon means "all columns".
print("First row of raster:", raster[0, :])

In [ ]:
# A whole column. Here the bare colon means "all rows".
print("First column of raster:", raster[ :,0])

In [ ]:

# The last two values of the second-to-last row
print("Last two elements of the second-to-last row:", raster[-2,-2:])

We can also do mathematical operations quickly.

In [ ]:

# Array operations happen element-wise: every single value in the
# array has 1 subtracted from it, with no loop needed.
raster_sum = raster - 1
print("Array after subtracting 1:", raster_sum)



This looks bizarre, but there is an explanation. The `uint8` data type in NumPy is an 8-bit **unsigned** integer, which can only hold values from 0 to 255. Being unsigned, it cannot represent negative numbers at all.

We chose `dtype=np.uint8` ourselves when we created the raster. So subtracting 1 from a pixel whose value is 0 does not give -1 - instead the value "wraps around" to 255. To allow negative values, we have to change the data type.

> ⚠️ **Common Pitfall: integer overflow with `uint8` raster data**
>
> This is a genuine trap in raster work. A `uint8` pixel can only hold values from 0 to 255, and when you calculate with a whole array, NumPy does **not** warn you when a result falls outside that range - it silently wraps around (for single values you may see a `RuntimeWarning`, but the result still wraps):
>
> ```python
> np.uint8(0) - 1     # gives 255, not -1
> np.uint8(255) + 1   # gives 0, not 256
> ```
>
> Nothing crashes, so a brightness adjustment or a band-difference calculation can quietly produce nonsense. If your arithmetic might go below 0 or above 255, convert the array first with `.astype(np.int16)` or `.astype(np.float64)`.

In [ ]:
# astype() makes a new array with a different data type.
# int16 is SIGNED, so it can hold negative numbers.
raster_signed = raster.astype(np.int16)  # or np.int32

In [ ]:

# Now subtracting 1 from 0 correctly gives -1
raster_sum = raster_signed - 1
print("Array after subtracting 1:", raster_sum)



Works now!

In [ ]:
raster_product = raster_signed * 2
print("Array after multiplying by 2:", raster_product)

### Array Manipulations

Reshaping an array in NumPy means changing the shape or structure of the array without changing its data.

In [ ]:

# Reshaping an array: the raster has 100 x 100 = 10000 values,
# and 5000 x 2 is also 10000, so this rearrangement is allowed.
reshaped_raster = raster.reshape((5000, 2))
print("Reshaped raster:\n", reshaped_raster)


In [ ]:
# The original raster is unchanged - reshape() returns a new arrangement
print("Raster:\n", raster)

In [ ]:

# Another valid reshape: 2 x 5000 is also 10000 values
reshaped_raster = raster.reshape((2, 5000))
print("Reshaped raster:\n", reshaped_raster)


> ⚠️ **Common Pitfall: `reshape()` must keep the same number of elements**
>
> You can rearrange an array into any shape you like, as long as the **total** number of values stays identical. Our raster holds 100 x 100 = 10000 values, so:
>
> ```python
> raster.reshape((5000, 2))   # fine:  5000 x 2 = 10000
> raster.reshape((2, 5000))   # fine:  2 x 5000 = 10000
> raster.reshape((100, 99))   # ValueError: 9900 is not 10000
> ```
>
> If you see `ValueError: cannot reshape array of size ... into shape ...`, multiply your requested dimensions together and compare the result with `array.size`.

## Combining and Splitting Arrays

In [ ]:
# A reminder of what array_1d contains
array_1d

In [ ]:
# Concatenating joins two arrays end to end.
# Note the double parentheses: the two arrays are passed together as one tuple.
array_concat = np.concatenate((array_1d, array_1d))

In [ ]:
# The result is one array of 10 values
array_concat

In [ ]:
# Splitting an array into multiple arrays.
# [2, 4] means "cut before index 2 and before index 4", giving three pieces.
array_split = np.split(array_1d, [2, 4])
print("Split Arrays:", array_split)

The second argument [2, 4] specifies the indices where the array should be split. For instance if we want to split the array after the first element the code would change to:

In [ ]:
# A single cut point gives two pieces: the first element, and the rest
array_split = np.split(array_1d, [1])
print("Split Arrays:", array_split)

We can also copy arrays very efficiently with np.copy.

In [ ]:

# .copy() makes an independent copy of the array.
# Changing the copy leaves the original untouched.
array_copy = array_1d.copy()
array_copy[0] = 100
print("Original Array:", array_1d)
print("Modified Copy:", array_copy)

> ⚠️ **Common Pitfall: a slice is a *view*, not a copy**
>
> This is one place where NumPy behaves differently from an ordinary Python list. Slicing a NumPy array does **not** hand you a separate copy - it gives a *view* onto the same underlying data. Changing the slice therefore changes the original:
>
> ```python
> part = raster[0:3, 0:70]
> part[0, 0] = 42          # raster[0, 0] is now 42 as well!
> ```
>
> That is exactly why `.copy()` exists, as used above. If you want to modify part of an array without touching the original, take a copy first: `part = raster[0:3, 0:70].copy()`.

### Other arithmetic operations

#### Summing

In [ ]:
# np.sum() adds up every value in the array
total_sum = np.sum(array_1d)
print("Sum of elements in array_1d:", total_sum)


In [ ]:
# The same function works on a 2D array, adding up all 10000 pixels
total_sum = np.sum(raster)
print("Sum of elements in raster:", total_sum)


#### Descriptive statistics

In [ ]:
# Finding the mean (average) of the array
mean_value = np.mean(array_1d)
print("Mean of array_1d:", mean_value)

In [ ]:
# Finding the smallest and largest values
min_value = np.min(array_1d)
max_value = np.max(array_1d)
print("Min:", min_value, "Max:", max_value)

#### Trigonometric Functions

In [ ]:

# Trigonometric functions are applied to every element at once.
# Note that NumPy works in radians, not degrees.
array_sin = np.sin(array_1d)
print("Sine of array_1d:", array_sin)

### Thresholding

Let's create a numpy array and use a value around the middle as a threshold.

In [ ]:
import numpy as np

# Example array (imagine this as a small greyscale image,
# where each number is a pixel brightness from 0 to 255)
image = np.array([[100, 150, 200],
                  [50, 130, 180],
                  [90, 170, 255]])

# Define the threshold: anything below this becomes black, anything else white
threshold = 128


Now we can manipulate the image values using the `np.where()` function. 

In [ ]:

# np.where(condition, value_if_true, value_if_false) checks every pixel at once
thresholded_image = np.where(image < threshold, 0, 255)

print("Original Image:\n", image)
print("Thresholded Image:\n", thresholded_image)


This example is similar to boolean logic or if statements, but it leverages NumPy's vectorized operations, which allow you to apply conditions to entire arrays at once instead of using explicit loops or multiple if statements! In a traditional way, the task above could be solved with multiple if statements.

In [ ]:
# The same result written the long way, using loops and if-else.
# This is much slower, and much longer, than the single np.where() line above.
thresholded_image = np.zeros_like(image)  # Create an empty array of the same shape

for i in range(image.shape[0]):  # Loop over rows
    for j in range(image.shape[1]):  # Loop over columns
        if image[i, j] < 128:
            thresholded_image[i, j] = 0
        else:
            thresholded_image[i, j] = 255

print(thresholded_image)



## References for Further Learning
- [NumPy Documentation](https://numpy.org/doc/stable/)
- [SciPy Lecture Notes on NumPy](https://lectures.scientific-python.org/intro/numpy/index.html)


## Summary

This final lesson introduced **NumPy**, the library that underpins pandas and most scientific computing in Python. For geomatics it is especially useful for rasters and other grids of numbers. You can now:

- **Create arrays** with `np.array()` from a list, or build empty ones with `np.zeros((rows, cols))`.
- **Index and slice** with `array[row, column]`, using `:` for "everything" along an axis, and negative indices to count from the end.
- **Check and change the data type** with `.dtype` and `.astype()` - which matters enormously for raster values.
- **Do element-wise maths** on a whole array at once (`raster - 1`, `raster * 2`), with no loop required.
- **Reshape, concatenate, and split** arrays with `.reshape()`, `np.concatenate()`, and `np.split()`.
- **Take a genuine copy** with `.copy()`.
- **Compute statistics** with `np.sum()`, `np.mean()`, `np.min()`, and `np.max()`, and apply functions like `np.sin()` to every element.
- **Threshold or classify** an entire array in a single line with `np.where(condition, value_if_true, value_if_false)`.

| Watch out for | Reminder |
| --- | --- |
| `uint8` arithmetic | Values wrap around silently (`0 - 1` becomes 255). Convert with `.astype()` first. |
| Slices | A slice is a **view**, so changing it changes the original. Use `.copy()` if you don't want that. |
| `reshape()` | The total number of elements must stay the same, or you get a `ValueError`. |
| `np.sin()` and friends | Trigonometric functions expect **radians**, not degrees. |

**Well done - this is the end of the lecture material!** You began by using Python as a calculator, and you can now read real datasets, clean them, analyse them, visualise them, and process raster grids. That is a genuinely useful toolkit for geomatics work.